# EAGER / PMBios — figure generation notebook

Every data panel in the manuscript, in figure order, from the source workbook or ImageJ
export through to the SVG/PNG placed in Illustrator.
Panel letters follow **ZZ_Manuscript_revised_V12.docx** / **Supplementary_V4.docx**.

Combines: `EAGER_figures.py`, `eager_fig4_fig5.py`, `make_fig4b*.py`, `make_fig4d.py`,
`make_fig5b.py`, `make_fig5d.py`, `make_fig4b_resample*.py`, `make_power_fig4b.py`.

| Panel | Section | Input | Output stem |
|---|---|---|---|
| 2a probe-ratio optimization | 2 | `072425_PROBE3_CONCENTRATION.xlsx` [plots] | `fig2a_probe_ratio` |
| 2b target-DNA titration | 2 | `080125_PROBE3_TARGET_LOW.xlsx` [Plots] | `fig2b_target_titration` |
| 2c mismatch specificity | 2 | `081325_PROBE3_MISMATCH.xlsx` [Plots] | `fig2c_mismatch` |
| 2d combinatorial heatmap | 2 | `102125ALLPROBES_Correct.xlsx` [Heatmap] | `fig2d_heatmap` |
| 4b on-chip FRET validation | 4 | `Fig4b.csv` → `figure4_plots/fig4b_source_data.csv` | `fig4b_onchip_validation` |
| 4d plasma protection | 4 | `Fig4_plasma_test_*.xlsx` → `fig4d_source_data.csv` | `fig4d_plasma_protection` |
| 5b probe set 2, 50 µM | 5 | `Probe 2/{1h,24h}/*.csv` → `fig5b_source_data.csv` | `fig5b_probeset2_50uM` |
| 5d probe set 3, 50 µM | 5 | `Probe 3/{1h,24h}/*.csv` → `fig5d_source_data.csv` | `fig5d_probeset3_50uM` |
| S3 target-fraction linearity | S3 | `082125_PROBE3_PRECENTAGE.xlsx` (values inline) | `figS3_target_fraction` |
| S5 subsampling, quenching | S5/S6 | `Fig4b.csv` (raw) | `figS5_subsampling_FP_vs_FPQP` |
| S6 subsampling, recovery | S5/S6 | `Fig4b.csv` (raw) | `figS6_subsampling_FPQP_vs_target` |
| S8 pumping characterization | S8 | `EAGER Data (from Prism).xlsx` [Volume per cycle nL] | `figS8_volume_per_cycle` |

Figures 1, 3 and Supplementary S1, S2, S4, S7 are Illustrator schematics or photographs —
no code.

**Run order:** section 1 (setup) first, then any panel section independently.
A panel whose input file is not in the folder prints a SKIPPED notice instead of failing,
so the notebook always runs end to end.

---
## 1 · Setup — paths, style, statistics, loaders

In [ ]:
"""Configuration. Edit BASE if the notebook is not in the project folder."""
import csv, glob, json, math, warnings
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  registers the 3D projection

try:
    import openpyxl
except ImportError:
    openpyxl = None
    print("openpyxl not installed - the Figure 2 and S8 sections will be skipped.")

BASE      = Path.cwd()                     # project folder
FIG2_DIR  = BASE / "figure2_plots"
FIG4_DIR  = BASE / "figure4_plots"
FIG5_DIR  = BASE / "figure5_plots"
SUPP_DIR  = BASE / "supp_plots"
PROBE_DIR = BASE                           # holds "Probe 2" / "Probe 3"
OUT_DIR   = BASE / "figures_notebook"      # everything written here; nothing is overwritten
OUT_DIR.mkdir(exist_ok=True)

N_BG = 3          # background ROIs at the END of each ImageJ export

print("project :", BASE)
print("outputs :", OUT_DIR)

In [ ]:
"""Shared style. Keeps Arial live in the SVG so Illustrator can edit the text."""
NPG   = ['#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F',
         '#8491B4', '#91D1C2', '#DC0000', '#7E6148', '#B09C85']
GRAY, TEAL = '#A6A6A6', '#00BFB3'          # Fig 5 bar key: no target / + target

plt.rcParams.update({
    "font.family": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 9, "axes.linewidth": 0.9,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.width": 0.9, "ytick.major.width": 0.9,
    "xtick.major.size": 3.5, "ytick.major.size": 3.5,
    "svg.fonttype": "none",                # text stays text in the SVG
    "pdf.fonttype": 42,
    "axes.titlesize": 10, "axes.titleweight": "bold",
})
warnings.filterwarnings("ignore", message="Glyph .* missing from font")
# Arial is absent on many Linux boxes; fall back quietly instead of one warning per glyph.
import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)


def save(fig, stem, out=None):
    """Write one SVG (vector, Illustrator-editable) + one 300-dpi PNG."""
    out = Path(out or OUT_DIR)
    fig.savefig(out / f"{stem}.svg", bbox_inches="tight", transparent=True)
    fig.savefig(out / f"{stem}.png", dpi=300, bbox_inches="tight")
    print("saved", stem + ".svg/.png")

In [ ]:
"""Statistics. Pure numpy - no scipy needed (the lab VM does not have it).

t-distribution tail from the regularized incomplete beta function; agrees with
scipy.stats to ~4 significant figures.
"""
def _betacf(a, b, x, itmax=300, eps=3e-16):
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c, d = 1.0, 1.0 - qab * x / qap
    d = 1.0 / (d if abs(d) > 1e-300 else 1e-300)
    h = d
    for m in range(1, itmax + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d; d = 1.0 / (d if abs(d) > 1e-300 else 1e-300)
        c = 1.0 + aa / c; c = c if abs(c) > 1e-300 else 1e-300
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d; d = 1.0 / (d if abs(d) > 1e-300 else 1e-300)
        c = 1.0 + aa / c; c = c if abs(c) > 1e-300 else 1e-300
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < eps:
            break
    return h


def _betainc(a, b, x):
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    lbeta = math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b)
    front = math.exp(lbeta + a * math.log(x) + b * math.log1p(-x))
    if x < (a + 1.0) / (a + b + 2.0):
        return front * _betacf(a, b, x) / a
    return 1.0 - math.exp(lbeta + b * math.log1p(-x) + a * math.log(x)) * _betacf(b, a, 1.0 - x) / b


def t_sf2(t, df):
    """Two-tailed p value for Student's t."""
    if not np.isfinite(t) or df <= 0: return float("nan")
    return _betainc(0.5 * df, 0.5, df / (df + t * t))


def welch(a, b):
    """Welch's unequal-variance two-sample t-test -> (t, df, p)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    se2 = va / na + vb / nb
    t = (a.mean() - b.mean()) / math.sqrt(se2)
    df = se2 ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    return t, df, t_sf2(t, df)


def paired(a, b):
    """Two-tailed paired t-test on matched observations -> (t, df, p)."""
    d = np.asarray(a, float) - np.asarray(b, float)
    n = len(d)
    t = d.mean() / (d.std(ddof=1) / math.sqrt(n))
    return t, n - 1, t_sf2(t, n - 1)


def stars(p):
    if not np.isfinite(p): return "?"
    return ("****" if p < 1e-4 else "***" if p < 1e-3 else
            "**" if p < 1e-2 else "*" if p < 0.05 else "n.s.")


def describe(v):
    v = np.asarray(v, float)
    return dict(n=len(v), mean=v.mean(), sd=v.std(ddof=1))


def mad_filter(v, thresh=3.5):
    """Legacy cleaner used by the 2026-07 Figure 2/5 scripts. Kept for reference;
    the Figure 4/5 panels below use per-image background subtraction instead."""
    v = np.asarray(v, float)
    med = np.median(v)
    mad = np.median(np.abs(v - med)) or 1.0
    return v[np.abs(v - med) / (1.4826 * mad) < thresh]

print("statistics ready (welch, paired, stars); scipy not required")

In [ ]:
"""Loaders.

Three input shapes appear in this project:
  A. Prism/Excel summary sheets     -> load_sheet()
  B. raw ImageJ Results exports     -> imagej_means() / background_correct() / imagej_blocks()
  C. archived per-spot source data  -> load_fig4b/4d/5 (already background-corrected)

Quantification used by every Figure 4/5 panel:
    B_min = min_j b_j        lowest background ROI, one value per IMAGE
    I_i   = mu_i - B_min     background-corrected intensity of spot i
    F     = mean(I | +target) / mean(I | no target)
"""
class MissingInput(Exception):
    """Raised when a source file is not in the folder; sections catch it and skip."""


def find_file(pattern, root=None):
    hits = sorted(Path(root or BASE).glob(pattern))
    if not hits:
        raise MissingInput(f"no file matching {pattern!r} in {root or BASE}")
    return hits[0]


def load_sheet(pattern, sheet=None, root=None):
    """Rows of an Excel sheet as plain lists (values only)."""
    if openpyxl is None:
        raise MissingInput("openpyxl is not installed")
    wb = openpyxl.load_workbook(find_file(pattern, root), data_only=True)
    ws = wb[sheet] if sheet else wb.active
    return [list(r) for r in ws.iter_rows(values_only=True)
            if any(c is not None for c in r)]


def imagej_means(fp, delimiter=","):
    """The ImageJ 'Mean' column (3rd) of a Results export, in file order."""
    vals = []
    with open(fp, newline="") as fh:
        rdr = csv.reader(fh, delimiter=delimiter)
        next(rdr, None)
        for row in rdr:
            if len(row) >= 3 and str(row[2]).strip():
                try: vals.append(float(row[2]))
                except ValueError: pass
    return np.array(vals)


def background_correct(fp, n_bg=N_BG, delimiter=","):
    """I_i = mu_i - B_min, with the LAST n_bg rows taken as background ROIs."""
    v = imagej_means(fp, delimiter)
    spots, bg = v[:-n_bg], v[-n_bg:]
    return spots - bg.min(), bg.min()


def imagej_blocks(fp, col0):
    """One column block of the multi-block Fig4b.csv export.

    Rows after a cell labelled 'Background' in the block's label column are
    background ROIs. Returns (spot means, background means), uncorrected.
    """
    rows = list(csv.reader(open(fp)))
    sp, bg, mode = [], [], 'sp'
    for r in rows[1:]:
        if len(r) <= col0 + 2: continue
        lab = r[col0].strip()
        if lab.lower().startswith('background'): mode = 'bg'; continue
        if lab == '': continue
        try: m = float(r[col0 + 2])
        except ValueError: continue
        (sp if mode == 'sp' else bg).append(m)
    return np.array(sp), np.array(bg)


def _numbered_rows(path):
    """Data rows of an archived source-data CSV (skips blanks and the B_min footer)."""
    out = []
    for r in list(csv.reader(open(path)))[1:]:
        if not r or not r[0].strip(): continue
        try: int(r[0])
        except ValueError: continue
        out.append(r)
    return out


def load_source_columns(path):
    """Archived source data -> {column header without the I_/_au wrapper: array}.

    Selected BY HEADER NAME, so this survives the layout change of 2026-09-03
    when the +target columns were added to fig4b_source_data.csv.
    """
    path = Path(path)
    if not path.exists(): raise MissingInput(f"{path.name} not found")
    hdr = list(csv.reader(open(path)))[0]
    rows = _numbered_rows(path)
    out = {}
    for i, h in enumerate(hdr[1:], start=1):
        name = h.strip()
        if name.startswith("I_"): name = name[2:]
        if name.endswith("_au"): name = name[:-3]
        out[name] = np.array([float(r[i]) for r in rows if len(r) > i])
    return out


def load_fig5(path):
    """timepoint, condition, spot, I_au -> {(timepoint, condition): array}."""
    path = Path(path)
    if not path.exists(): raise MissingInput(f"{path.name} not found")
    out = {}
    for r in list(csv.reader(open(path)))[1:]:
        if len(r) < 4 or r[1] not in ("no target", "+ target"): continue
        out.setdefault((r[0], r[1]), []).append(float(r[3]))
    return {k: np.array(v) for k, v in out.items()}


RESULTS = {}     # every section drops its numbers here for the summary table at the end
print("loaders ready")

---
## 2 · Figure 2 — liquid-phase optimization and specificity

Probe set 3 as the model. Bars = mean, error bars = s.d. (ddof = 1).

Source workbooks live with the plate-reader exports; if they are not in this folder each
cell prints a SKIPPED notice. Copy them next to the notebook to regenerate the panels.

> Panel letters here are the **V12** ones (a–d). The old `EAGER_figures.py` named the same
> panels b–e, from a layout whose panel a was the workflow schematic — that schematic is now
> Supplementary Figure S2.

### Figure 2a — probe-ratio optimization
`072425_PROBE3_CONCENTRATION.xlsx` → sheet `plots`, columns: fluorescent probe (µM),
quencher probe (µM), mean, s.d. → 3-D bar grid; the three ratios carried forward
(Q/F 0.5/0.5, 1/0.5, 1/1) are highlighted in red.

In [ ]:
try:
    rows = load_sheet("072425_PROBE3_CONCENTRATION.xlsx", "plots")[1:]
    FP = sorted(set(r[0] for r in rows)); QP = sorted(set(r[1] for r in rows))
    val = {(r[0], r[1]): (r[2], r[3]) for r in rows}
    opt = {(0.5, 0.5), (0.5, 1), (1, 1)}
    fpcol = ['#8491B4', '#91D1C2', '#4DBBD5', '#3C5488']

    fig = plt.figure(figsize=(6.2, 5.2)); ax = fig.add_subplot(111, projection='3d')
    dx = dy = 0.5; cap = 0.26
    for xi, fp in enumerate(FP):
        for yi, qp in enumerate(QP):
            m, s = val[(fp, qp)]; hi = (fp, qp) in opt
            ax.bar3d(xi - dx/2, yi - dy/2, 0, dx, dy, m, shade=True,
                     alpha=(0.85 if hi else 0.45),
                     color=('#E64B35' if hi else fpcol[xi]),
                     edgecolor=('#7a1a12' if hi else 'white'),
                     linewidth=(1.2 if hi else 0.3))
    for xi, fp in enumerate(FP):                      # error bars on top of the bars
        for yi, qp in enumerate(QP):
            m, s = val[(fp, qp)]
            ax.plot([xi, xi], [yi, yi], [m, m + s], color='black', lw=1.7, zorder=10)
            ax.plot([xi - cap/2, xi + cap/2], [yi, yi], [m + s, m + s],
                    color='black', lw=1.7, zorder=10)

    ax.set_xticks(range(len(FP))); ax.set_xticklabels(FP)
    ax.set_yticks(range(len(QP))); ax.set_yticklabels(QP)
    ax.set_xlabel("Fluorescent probe (µM)", labelpad=8)
    ax.set_ylabel("Quencher probe (µM)", labelpad=8)
    ax.set_zlabel("Mean intensity (a.u.)", labelpad=6)
    ax.view_init(elev=22, azim=45)
    for pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
        pane.set_alpha(0); pane.set_edgecolor('0.85')
    ax.grid(True, color='0.9', linewidth=0.5)
    ax.legend(handles=[Patch(facecolor='#E64B35', edgecolor='#7a1a12', linewidth=1,
                             alpha=0.85, label='Optimized ratios (Q/F 0.5/0.5, 1/0.5, 1/1)')],
              loc='upper left', frameon=False, fontsize=8, bbox_to_anchor=(0.0, 0.98))
    ax.set_title("a  Probe-ratio optimization", loc='left', pad=0)
    save(fig, "fig2a_probe_ratio"); plt.show()
except MissingInput as e:
    print("SKIPPED Figure 2a -", e)

### Figure 2b — target-DNA titration
`080125_PROBE3_TARGET_LOW.xlsx` → sheet `Plots`, columns: ratio, target DNA (µM), mean, s.d.
Three selected ratios × target 0–0.1 µM.

In [ ]:
try:
    rows = load_sheet("080125_PROBE3_TARGET_LOW.xlsx", "Plots")[1:]
    ratios = ['0.5/0.5', '1/0.5', '1/1']
    targets = sorted(set(round(r[1], 4) for r in rows))
    val = {(r[0], round(r[1], 4)): (r[2], r[3]) for r in rows}
    rcol = {'0.5/0.5': '#3C5488', '1/0.5': '#4DBBD5', '1/1': '#00A087'}

    fig = plt.figure(figsize=(6.8, 5.2)); ax = fig.add_subplot(111, projection='3d')
    dx = dy = 0.5; cap = 0.26
    for xi, rt in enumerate(ratios):
        for yi, tg in enumerate(targets):
            m, s = val[(rt, tg)]
            ax.bar3d(xi - dx/2, yi - dy/2, 0, dx, dy, m, shade=True, alpha=0.5,
                     color=rcol[rt], edgecolor='white', linewidth=0.3)
    for xi, rt in enumerate(ratios):
        for yi, tg in enumerate(targets):
            m, s = val[(rt, tg)]
            ax.plot([xi, xi], [yi, yi], [m, m + s], color='black', lw=1.7, zorder=10)
            ax.plot([xi - cap/2, xi + cap/2], [yi, yi], [m + s, m + s],
                    color='black', lw=1.7, zorder=10)

    ax.set_xticks(range(len(ratios))); ax.set_xticklabels(ratios)
    ax.set_yticks(range(len(targets))); ax.set_yticklabels([f"{t:g}" for t in targets])
    ax.set_xlabel("Quencher/Fluorescent (µM/µM)", labelpad=10)
    ax.set_ylabel("Target DNA (µM)", labelpad=10)
    ax.set_zlabel("Mean intensity (a.u.)", labelpad=6)
    ax.view_init(elev=28, azim=-52); ax.invert_yaxis()
    for pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
        pane.set_alpha(0); pane.set_edgecolor('0.85')
    ax.grid(True, color='0.9', linewidth=0.5)
    ax.set_title("b  Target-DNA titration", loc='left', pad=0)
    save(fig, "fig2b_target_titration"); plt.show()
except MissingInput as e:
    print("SKIPPED Figure 2b -", e)

### Figure 2c — mismatch specificity
`081325_PROBE3_MISMATCH.xlsx` → sheet `Plots`, columns: label, mean, s.d.
Rows ordered all-match → 2/4/6 bp inside the quenching region → 2/4/6 bp outside it.

In [ ]:
try:
    rows = load_sheet("081325_PROBE3_MISMATCH.xlsx", "Plots")[1:]
    rr = [(str(r[0]).strip(), r[1], r[2]) for r in rows]
    allm = [x for x in rr if x[0] == "All Match"][0]
    mm = [x for x in rr if x[0] in ("2 bp", "4 bp", "6 bp")]
    order = [allm] + mm[:3] + mm[3:6]
    means = [x[1] for x in order]; sds = [x[2] for x in order]
    xlabs = ["All\nmatch", "2 bp", "4 bp", "6 bp", "2 bp", "4 bp", "6 bp"]
    cols = [NPG[0]] + [NPG[5]]*3 + [NPG[2]]*3
    xpos = [0, 1.3, 2.1, 2.9, 4.2, 5.0, 5.8]

    fig, ax = plt.subplots(figsize=(5.4, 3.5))
    ax.bar(xpos, means, yerr=sds, width=0.72, color=cols, edgecolor='black', linewidth=0.6,
           error_kw=dict(elinewidth=0.9, capsize=2.5, capthick=0.9, ecolor='black'))
    ax.set_xticks(xpos); ax.set_xticklabels(xlabs)
    ax.set_ylabel("Mean intensity (a.u.)")
    ytop = max(m + s for m, s in zip(means, sds)); ax.set_ylim(0, ytop * 1.22)

    def bracket(x0, x1, y, txt, color):
        ax.plot([x0, x0, x1, x1], [y, y + 8, y + 8, y], color=color, lw=1.0)
        ax.text((x0 + x1) / 2, y + 12, txt, ha='center', va='bottom', fontsize=8, color=color)

    bracket(1.3, 2.9, ytop * 1.03, "Mismatch in\nquenching region", NPG[5])
    bracket(4.2, 5.8, ytop * 1.03, "Mismatch outside\nquenching region", NPG[2])
    ax.set_title("c  Mismatch specificity", loc='left')
    save(fig, "fig2c_mismatch"); plt.show()
except MissingInput as e:
    print("SKIPPED Figure 2c -", e)

### Figure 2d — combinatorial specificity heatmap
`102125ALLPROBES_Correct.xlsx` → sheet `Heatmap`. Each target row is divided by the
`Buffer` row, so the matrix is **fold change over buffer**; on-diagonal 7.2–10.6,
off-diagonal near 1.

In [ ]:
try:
    rows = load_sheet("102125ALLPROBES_Correct.xlsx", "Heatmap")
    dd = {str(r[0]): r[1:4] for r in rows[1:]}
    buf = np.array(dd["Buffer"], float)
    tgts = ["Target1", "Target2", "Target3"]
    M = np.array([np.array(dd[t], float) / buf for t in tgts])

    cmap = LinearSegmentedColormap.from_list("npg_blue",
                                             ["#FFFFFF", "#B9D3E6", "#4DBBD5", "#3C5488"])
    fig, ax = plt.subplots(figsize=(4.3, 3.6))
    im = ax.imshow(M, cmap=cmap, vmin=0, vmax=np.ceil(M.max()))
    ax.set_xticks(range(3)); ax.set_xticklabels([f"Probe {i+1}" for i in range(3)])
    ax.set_yticks(range(3)); ax.set_yticklabels([f"Target {i+1}" for i in range(3)])
    for i in range(3):
        for j in range(3):
            v = M[i, j]
            ax.text(j, i, f"{v:.2f}", ha='center', va='center',
                    color='white' if v > M.max() * 0.55 else '#222222',
                    fontsize=10, fontweight='bold')
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Fold change over buffer", fontsize=9); cb.outline.set_linewidth(0.8)
    ax.set_xticks(np.arange(-.5, 3, 1), minor=True); ax.set_yticks(np.arange(-.5, 3, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=2); ax.tick_params(which='minor', length=0)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title("d  Combinatorial specificity", loc='left')
    RESULTS["fig2d_matrix"] = M
    save(fig, "fig2d_heatmap"); plt.show()
except MissingInput as e:
    print("SKIPPED Figure 2d -", e)

---
## 4 · Figure 4 — on-chip validation and plasma protection

Both panels read the archived per-spot source data in `figure4_plots/`, which is already
background-corrected (`I_i = mu_i − B_min`, B_min = lowest of the off-spot background ROIs of
that image; the B_min values are kept in the footer row of each CSV).

To rebuild the source data from raw ImageJ exports instead, use `background_correct()` /
`imagej_blocks()` from section 1 — see the "raw rebuild" cell at the end of this section.

### Figure 4b — on-chip validation of the solid-state FRET assay

Conditions are selected **by column header**, so this works with both the original 3-column
source data and the 4-condition version written on 2026-09-03
(`spot, I_FP, I_FP+Target, I_FP+FQ, I_FP+FQ+Target`). `FP+FQ` and `FP+QP` are the same thing.

In [ ]:
ALIAS = {"FP": "FP", "FP+Target": "FP+Target",
         "FP+FQ": "FP+QP", "FP+QP": "FP+QP",
         "FP+FQ+Target": "FP+QP+Target", "FP+QP+Target": "FP+QP+Target"}

def load_fig4b(path=None):
    cols = load_source_columns(path or FIG4_DIR / "fig4b_source_data.csv")
    return {ALIAS.get(k, k): v for k, v in cols.items()}

def analyse_fig4b(d):
    out = {"desc": {k: describe(v) for k, v in d.items()}}
    for key, (a, b) in {"FP_vs_FPQP": ("FP", "FP+QP"),
                        "FPQP_vs_T":  ("FP+QP", "FP+QP+Target"),
                        "FP_vs_T":    ("FP", "FP+QP+Target")}.items():
        t, df, p = welch(d[a], d[b])
        out[key] = dict(t=t, df=df, p=p, stars=stars(p))
    out["quench_fold"]            = d["FP"].mean() / d["FP+QP"].mean()
    out["recovery_fold"]          = d["FP+QP+Target"].mean() / d["FP+QP"].mean()
    out["residual_pct_of_target"] = 100 * d["FP+QP"].mean() / d["FP+QP+Target"].mean()
    return out

try:
    d4b = load_fig4b()
    r4b = analyse_fig4b(d4b)
    RESULTS["4b"] = r4b
    for k, s in r4b["desc"].items():
        print(f"{k:>14s}  n={s['n']:2d}  mean={s['mean']:9.1f}  s.d.={s['sd']:8.1f}")
    for key in ("FP_vs_FPQP", "FPQP_vs_T", "FP_vs_T"):
        s = r4b[key]
        print(f"{key:>12s}  Welch t={s['t']:.3f}  df={s['df']:.1f}  p={s['p']:.3g}  {s['stars']}")
    print(f"quenching {r4b['quench_fold']:.2f}x   recovery {r4b['recovery_fold']:.2f}x   "
          f"residual = {r4b['residual_pct_of_target']:.0f}% of the +target signal")
except MissingInput as e:
    print("SKIPPED Figure 4b -", e)

In [ ]:
"""Figure 4b, three-bar version (FP / FP+QP / FP+QP+Target)."""
try:
    order   = ["FP", "FP+QP", "FP+QP+Target"]
    labels  = ["Fluorescent Probes",
               "Fluorescent Probes\nQuencher Probes",
               "Fluorescent Probes\nQuencher Probes\nTarget"]
    colours = [NPG[3], NPG[5], NPG[0]]
    means = [d4b[k].mean() for k in order]
    sds   = [d4b[k].std(ddof=1) for k in order]

    fig, ax = plt.subplots(figsize=(3.9, 4.2))
    ax.bar(range(3), means, yerr=sds, width=0.62, color=colours, edgecolor="black",
           linewidth=0.6,
           error_kw=dict(elinewidth=0.9, capsize=2.5, capthick=0.9, ecolor="black"), zorder=3)
    for i, k in enumerate(order):
        jit = np.random.RandomState(i).uniform(-0.11, 0.11, len(d4b[k]))
        ax.scatter(np.full(len(d4b[k]), i) + jit, d4b[k], s=9, color="black",
                   alpha=0.5, zorder=4, linewidths=0)

    ymax = max(max(m + s for m, s in zip(means, sds)), max(d4b[k].max() for k in order))
    ax.set_ylim(0, ymax * 1.42)
    for (i, j), key, lvl in (((0, 1), "FP_vs_FPQP", 0.06),
                             ((1, 2), "FPQP_vs_T",  0.16),
                             ((0, 2), "FP_vs_T",    0.28)):
        y = ymax * (1.02 + lvl)
        ax.plot([i, i, j, j], [y, y + ymax*0.02, y + ymax*0.02, y], color="black", lw=0.9)
        ax.text((i + j) / 2, y + ymax * 0.025, r4b[key]["stars"],
                ha="center", va="bottom", fontsize=10)

    ax.set_xticks(range(3)); ax.set_xticklabels(labels, fontsize=7.5)
    ax.set_ylabel("Background-corrected spot intensity (a.u.)")
    save(fig, "fig4b_onchip_validation"); plt.show()
except (MissingInput, NameError) as e:
    print("SKIPPED Figure 4b panel -", e)

In [ ]:
"""Figure 4b, four-bar version - matches the +/- condition table under the panel.

Bars: F+ Q- T- | F+ Q+ T- | F+ Q- T+ | F+ Q+ T+
Only runs when the source data carries the FP+Target column.
"""
try:
    need = ["FP", "FP+QP", "FP+Target", "FP+QP+Target"]
    if not all(k in d4b for k in need):
        raise MissingInput("source data has no FP+Target column - 3-bar version only")

    means = [d4b[k].mean() for k in need]
    sds   = [d4b[k].std(ddof=1) for k in need]
    cols  = [NPG[3], NPG[5], NPG[1], NPG[0]]

    fig, ax = plt.subplots(figsize=(4.8, 4.4))
    ax.bar(range(4), means, yerr=sds, width=0.66, color=cols, edgecolor="black",
           linewidth=0.7,
           error_kw=dict(elinewidth=1.0, capsize=3, capthick=1.0, ecolor="black"), zorder=3)
    for i, k in enumerate(need):
        jit = np.random.RandomState(i).uniform(-0.12, 0.12, len(d4b[k]))
        ax.scatter(np.full(len(d4b[k]), i) + jit, d4b[k], s=9, color="black",
                   alpha=0.45, zorder=4, linewidths=0)

    ymax = max(max(m + s for m, s in zip(means, sds)), max(d4b[k].max() for k in need))
    ax.set_ylim(0, ymax * 1.30)
    for (i, j), lvl in (((0, 1), 0.02), ((1, 3), 0.12)):
        t, df, p = welch(d4b[need[i]], d4b[need[j]])
        y = ymax * (1.02 + lvl)
        ax.plot([i, i, j, j], [y, y + ymax*0.02, y + ymax*0.02, y], color="black", lw=0.9)
        ax.text((i + j) / 2, y + ymax * 0.025, stars(p), ha="center", va="bottom", fontsize=10)

    ax.set_xticks(range(4))
    ax.set_xticklabels(["F +\nQ −\nT −", "F +\nQ +\nT −", "F +\nQ −\nT +", "F +\nQ +\nT +"],
                       fontsize=9)
    ax.set_ylabel("Background-corrected spot intensity (a.u.)")
    save(fig, "fig4b_onchip_validation_4bar"); plt.show()
except (MissingInput, NameError) as e:
    print("SKIPPED Figure 4b four-bar -", e)

### Figure 4d — plasma protection of the printed microarray

The **same spots** are imaged before and after plasma treatment, so the design is paired.
The manuscript reports the Welch (unpaired) result, which is the conservative choice; the
paired result is computed here as well so the decision stays visible.

In [ ]:
try:
    cols = load_source_columns(FIG4_DIR / "fig4d_source_data.csv")
    before, after = cols["before"], cols["after"]

    tw, dfw, pw = welch(before, after)
    tp, dfp, pp = paired(before, after)
    ratios = after / before
    r4d = {"desc": {"before": describe(before), "after": describe(after)},
           "welch":  dict(t=tw, df=dfw, p=pw, stars=stars(pw)),
           "paired": dict(t=tp, df=dfp, p=pp, stars=stars(pp)),
           "retention_perspot_pct": 100 * ratios.mean(),
           "retention_perspot_sd":  100 * ratios.std(ddof=1),
           "retention_ratio_of_means_pct": 100 * after.mean() / before.mean()}
    RESULTS["4d"] = r4d

    print(f"before  n={len(before):2d}  mean={before.mean():8.1f}  s.d.={before.std(ddof=1):7.1f}")
    print(f"after   n={len(after):2d}  mean={after.mean():8.1f}  s.d.={after.std(ddof=1):7.1f}")
    print(f"retention: {r4d['retention_perspot_pct']:.1f}% per spot "
          f"(s.d. {r4d['retention_perspot_sd']:.1f}), "
          f"{r4d['retention_ratio_of_means_pct']:.1f}% as ratio of means")
    print(f"Welch  t={tw:.3f} df={dfw:.1f} p={pw:.3g} {stars(pw)}   <- reported")
    print(f"paired t={tp:.3f} df={dfp:.0f} p={pp:.3g} {stars(pp)}")

    fig, ax = plt.subplots(figsize=(3.5, 4.2))
    means = [before.mean(), after.mean()]; sds = [before.std(ddof=1), after.std(ddof=1)]
    ax.bar([0, 1], means, yerr=sds, width=0.6, color=[NPG[3], NPG[1]], edgecolor="black",
           linewidth=0.6,
           error_kw=dict(elinewidth=0.9, capsize=2.5, capthick=0.9, ecolor="black"), zorder=3)
    for k, g in enumerate([before, after]):
        jit = np.random.RandomState(k).uniform(-0.11, 0.11, len(g))
        ax.scatter(np.full(len(g), k) + jit, g, s=9, color="black", alpha=0.5,
                   zorder=4, linewidths=0)
    ymax = max(means[0] + sds[0], means[1] + sds[1], before.max(), after.max())
    ax.set_ylim(0, ymax * 1.28)
    y = ymax * 1.10
    ax.plot([0, 0, 1, 1], [y, y + ymax*0.02, y + ymax*0.02, y], color="black", lw=0.9)
    ax.text(0.5, y + ymax * 0.03, r4d["welch"]["stars"], ha="center", va="bottom", fontsize=10)
    ax.text(1, means[1] + sds[1] + ymax * 0.03,
            f"{r4d['retention_perspot_pct']:.0f}%\nretained",
            ha="center", va="bottom", fontsize=8, color=NPG[3])
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Before plasma\ntreatment",
                        "After plasma\ntreatment\n(with protection)"])
    ax.set_ylabel("Background-corrected spot intensity (a.u.)")
    save(fig, "fig4d_plasma_protection"); plt.show()
except MissingInput as e:
    print("SKIPPED Figure 4d -", e)

In [ ]:
"""Optional: rebuild the Figure 4 source data from raw ImageJ exports.

Set RAW_DIR to the folder holding Fig4b.csv and Fig4_plasma_test_before/after.csv.
Column blocks of Fig4b.csv (two images of the same slide, same ROI set):
    no-target image : ROIs 1-18 = FP        (cols 12-16), ROIs 19-36 = FP+QP        (cols 17-21)
    +target image   : ROIs 1-18 = FP+Target (cols  0-4 ), ROIs 19-36 = FP+QP+Target (cols  5-9 )
Each image carries its own background ROIs, so B_min is taken per image.
"""
RAW_DIR = BASE / "raw_imagej"

def rebuild_fig4b_source(raw=None, write=False):
    src = Path(raw or RAW_DIR) / "Fig4b.csv"
    if not src.exists():
        raise MissingInput(f"{src} not found - keep using the archived source data")
    FPt, bgt = imagej_blocks(src, 0)      # FP + Target        (+target image)
    FQt, _   = imagej_blocks(src, 5)      # FP + QP + Target   (+target image)
    FP,  bgn = imagej_blocks(src, 12)     # FP                 (no-target image)
    FQ,  _   = imagej_blocks(src, 17)     # FP + QP            (no-target image)
    d = {"FP": FP - bgn.min(), "FP+QP": FQ - bgn.min(),
         "FP+Target": FPt - bgt.min(), "FP+QP+Target": FQt - bgt.min()}
    if write:
        p = OUT_DIR / "fig4b_source_data.csv"
        with open(p, "w", newline="") as fh:
            w = csv.writer(fh)
            w.writerow(["spot", "I_FP_au", "I_FP+Target_au", "I_FP+FQ_au", "I_FP+FQ+Target_au"])
            for i in range(len(d["FP"])):
                w.writerow([i + 1] + [f"{d[k][i]:.3f}" for k in
                                      ("FP", "FP+Target", "FP+QP", "FP+QP+Target")])
            w.writerow([]); w.writerow(["B_min_no_target_image", f"{bgn.min():.3f}",
                                        "B_min_target_image", f"{bgt.min():.3f}"])
        print("wrote", p)
    return d

try:
    _ = rebuild_fig4b_source()
    print("raw rebuild OK:", {k: len(v) for k, v in _.items()})
except MissingInput as e:
    print("raw rebuild not run -", e)

---
## 5 · Figure 5 — on-chip detection, two probe sets

50 µM target only (100 µM was dropped from the main figure). Probe set 2 on top (a, b),
probe set 3 below (c, d); panels **a** and **c** are fluorescence images placed in
Illustrator, **b** and **d** are these bar charts.

Source: `Probe {2,3}/{1h,24h}/Results Probe{N} (serie1) with {0,1} target 200ms.csv`
(serie1 = 50 µM), background-corrected per image, archived as `fig5{b,d}_source_data.csv`.

In [ ]:
def analyse_fig5(data):
    out = {}
    for t in ["1 h", "24 h"]:
        a, b = data[(t, "no target")], data[(t, "+ target")]
        tt, df, p = welch(a, b)
        out[t] = dict(no_target=describe(a), target=describe(b), fold=b.mean() / a.mean(),
                      t=tt, df=df, p=p, stars=stars(p),
                      residual_pct_of_target=100 * a.mean() / b.mean())
    return out


def panel_fig5(data, res, title, stem):
    times = ["1 h", "24 h"]
    centers, w = np.arange(len(times), dtype=float), 0.38
    ymax = max(v.max() for v in data.values())
    fig, ax = plt.subplots(figsize=(3.8, 4.2))
    for gi, t in enumerate(times):
        for k, (cond, col) in enumerate((("no target", GRAY), ("+ target", TEAL))):
            v = data[(t, cond)]
            x = centers[gi] + (k - 0.5) * w
            ax.bar(x, v.mean(), width=w, color=col, edgecolor="black", linewidth=0.5,
                   yerr=v.std(ddof=1),
                   error_kw=dict(elinewidth=0.9, capsize=2, capthick=0.9, ecolor="black"),
                   zorder=3)
            jit = np.random.RandomState(gi * 2 + k).uniform(-0.10, 0.10, len(v))
            ax.scatter(np.full(len(v), x) + jit, v, s=8, color="black", alpha=0.5,
                       zorder=4, linewidths=0)
        r = res[t]
        xa, xb = centers[gi] - 0.5 * w, centers[gi] + 0.5 * w
        yb = max(data[(t, "no target")].max(), data[(t, "+ target")].max()) + ymax * 0.06
        ax.plot([xa, xa, xb, xb], [yb, yb + ymax*0.02, yb + ymax*0.02, yb], color="black", lw=1.0)
        ax.text(centers[gi], yb + ymax * 0.03, f"{r['fold']:.1f}-fold  {r['stars']}",
                ha="center", va="bottom", fontsize=8.5)
    ax.set_xticks(centers); ax.set_xticklabels([f"50 µM\n{t}" for t in times])
    ax.set_ylabel("Background-corrected spot intensity (a.u.)")
    ax.set_ylim(0, ymax * 1.30)
    ax.legend(handles=[Patch(facecolor=GRAY, edgecolor="black", lw=0.5, label="No target"),
                       Patch(facecolor=TEAL, edgecolor="black", lw=0.5, label="+ Target")],
              frameon=False, fontsize=8, loc="upper left", ncol=2,
              columnspacing=1.0, handlelength=1.2)
    ax.set_title(title, loc="left")
    save(fig, stem)
    return fig


def report_fig5(panel, res):
    for t, r in res.items():
        print(f"{panel} {t:>4s}  no target n={r['no_target']['n']:2d} "
              f"mean={r['no_target']['mean']:7.1f} | + target n={r['target']['n']:2d} "
              f"mean={r['target']['mean']:7.1f} | fold={r['fold']:.2f}x  "
              f"p={r['p']:.3g} {r['stars']}  (residual {r['residual_pct_of_target']:.0f}% of target)")

In [ ]:
"""Figure 5b - probe set 2."""
try:
    d5b = load_fig5(FIG5_DIR / "fig5b_source_data.csv")
    r5b = analyse_fig5(d5b); RESULTS["5b"] = r5b
    report_fig5("5b", r5b)
    panel_fig5(d5b, r5b, "b  Probe set 2", "fig5b_probeset2_50uM"); plt.show()
except MissingInput as e:
    print("SKIPPED Figure 5b -", e)

In [ ]:
"""Figure 5d - probe set 3."""
try:
    d5d = load_fig5(FIG5_DIR / "fig5d_source_data.csv")
    r5d = analyse_fig5(d5d); RESULTS["5d"] = r5d
    report_fig5("5d", r5d)
    panel_fig5(d5d, r5d, "d  Probe set 3", "fig5d_probeset3_50uM"); plt.show()
except MissingInput as e:
    print("SKIPPED Figure 5d -", e)

In [ ]:
"""Optional: rebuild the Figure 5 source data straight from the ImageJ exports.

serie1 = 50 uM target, serie2 = 100 uM (not used in the main figure).
The last N_BG rows of each export are the background ROIs.
"""
def rebuild_fig5_source(probe="Probe 2", serie="serie1", write_to=None):
    out, bmins = {}, {}
    for t, label in (("1h", "1 h"), ("24h", "24 h")):
        for tgt, cond in (("0", "no target"), ("1", "+ target")):
            pat = (Path(PROBE_DIR) / probe / t /
                   f"Results {probe.replace(' ', '')} ({serie}) with {tgt} target 200ms.csv")
            hits = glob.glob(str(pat))
            if not hits:
                raise MissingInput(f"{pat} not found")
            I, bmin = background_correct(hits[0])
            out[(label, cond)] = I
            bmins[(label, cond)] = bmin
    if write_to:
        with open(write_to, "w", newline="") as fh:
            w = csv.writer(fh); w.writerow(["timepoint", "condition", "spot", "I_au"])
            for (t, c), v in out.items():
                for i, x in enumerate(v, 1):
                    w.writerow([t, c, i, f"{x:.3f}"])
        print("wrote", write_to)
    return out, bmins

try:
    _d, _b = rebuild_fig5_source("Probe 2", "serie1")
    print("probe set 2, 50 uM, rebuilt from ImageJ:",
          {k: (len(v), round(v.mean(), 1)) for k, v in _d.items()})
    print("B_min per image:", {k: round(v, 1) for k, v in _b.items()})
except MissingInput as e:
    print("raw rebuild not run -", e)

---
## S3 · Supplementary Figure S3 — target-fraction linearity

Fluorescence versus the fraction of 0.1 µM target DNA, 0–100 %.
Values are the background-subtracted replicate means from
`082125_PROBE3_PRECENTAGE.xlsx` (Sheet1); they are inlined here because the workbook is not
in the project folder. Trendline in the caption: y = 7.98x + 49.7, R² = 0.9934.

In [ ]:
frac = np.array([0, 0.2, 0.4, 0.6, 0.8, 1.0])
reps = np.array([
    [72.667, 217.667, 344.000, 475.667, 614.333, 860.667],
    [61.667, 203.000, 353.667, 515.333, 662.333, 874.667],
    [59.667, 222.000, 364.000, 529.333, 676.000, 890.667],
    [62.333, 222.333, 369.667, 518.000, 684.000, 914.333]])

means, sds, n = reps.mean(0), reps.std(0, ddof=1), reps.shape[0]
slope, intercept = np.polyfit(frac * 100, means, 1)
resid = means - (slope * frac * 100 + intercept)
r2 = 1 - resid.var() / means.var()
print(f"y = {slope:.2f}x + {intercept:.1f}   R2 = {r2:.4f}   (n = {n} replicates per point)")
RESULTS["S3"] = dict(slope=slope, intercept=intercept, r2=r2, n=n)

xpos = np.arange(len(frac))
fig, ax = plt.subplots(figsize=(4.6, 3.6))
ax.bar(xpos, means, width=0.66, color=TEAL, edgecolor='black', linewidth=0.6, yerr=sds,
       error_kw=dict(elinewidth=0.9, capsize=2.5, capthick=0.9, ecolor='black'), zorder=3)
for k in range(len(frac)):
    ax.scatter(np.full(n, k) + np.random.RandomState(k).uniform(-0.08, 0.08, n), reps[:, k],
               s=10, color='black', alpha=0.5, zorder=4, linewidths=0)
ax.plot(xpos, slope * frac * 100 + intercept, color='0.35', lw=1.0, ls=(0, (4, 3)), zorder=5)
ax.set_xticks(xpos); ax.set_xticklabels([f"{int(f*100)}" for f in frac])
ax.set_xlabel("Target fraction (%)"); ax.set_ylabel("Mean intensity (a.u.)")
ax.set_ylim(0, (means + sds).max() * 1.15)
ax.set_title("Target-fraction linearity", loc='left')
save(fig, "figS3_target_fraction"); plt.show()

---
## S5, S6 · Spot-subsampling — how many spots are enough?

Printed spots vary strongly within one array (CV ≈ 50–66 %), so the precision of a condition
mean is set by how many spots are averaged.

**Method.** For each n from 3 to N, draw n spots at random **without replacement** from each
condition, re-run Welch's two-sample t-test on that subset, and repeat 4 000 times
(`np.random.default_rng(0)`, so the figure is reproducible). Plot the p-value distribution
(violin, median, 95 % range) on a log axis against n, with the α = 0.05 line and the
percentage of draws reaching significance.

| Panel | Comparison | Reads |
|---|---|---|
| **S5** | Fluorescent probes vs + quencher probes (**quenching**) | Figure 4b spots |
| **S6** | FP+QP vs FP+QP+Target (**target recovery**) | Figure 4b spots |

In [ ]:
DRAWS = 4000            # lower to 500 for a quick pass
ALPHA = 0.05
FLOOR = 1e-5            # p values clipped here for the log axis only


def subsample_pvalues(A, B, draws=DRAWS, seed=0, nmin=3):
    """Welch p value distribution when only n spots per condition are used."""
    A, B = np.asarray(A, float), np.asarray(B, float)
    N = min(len(A), len(B))
    rng = np.random.default_rng(seed)
    ns, dist, frac = list(range(nmin, N + 1)), [], []
    for n in ns:
        p = np.empty(draws)
        for k in range(draws):
            i = rng.choice(len(A), n, replace=False)
            j = rng.choice(len(B), n, replace=False)
            p[k] = welch(A[i], B[j])[2]
        dist.append(np.clip(p, FLOOR, 1.0))
        frac.append((p < ALPHA).mean())
    return ns, dist, frac


def subsampling_panel(ns, dist, frac, title, stem, colour='#B7C0D8'):
    fig, ax = plt.subplots(figsize=(7.4, 4.6))
    ax.set_yscale('log')
    ax.axhspan(FLOOR, ALPHA, color=NPG[2], alpha=0.07, zorder=0)
    parts = ax.violinplot([np.log10(d) for d in dist], positions=ns, widths=0.85,
                          showextrema=False, showmedians=False)
    for pc in parts['bodies']:                       # violins are built in log10 space,
        pc.set_facecolor(colour); pc.set_edgecolor('none'); pc.set_alpha(0.75)
        v = pc.get_paths()[0].vertices               # then mapped onto the log axis
        v[:, 1] = 10 ** v[:, 1]
    for n, d in zip(ns, dist):
        lo, hi = np.percentile(d, [2.5, 97.5])
        ax.plot([n, n], [lo, hi], color='black', lw=1.2, zorder=3, solid_capstyle='butt')
        ax.plot([n], [np.median(d)], 'o', ms=4, color='black', zorder=4)
    ax.axhline(ALPHA, color='red', lw=1.4, ls=(0, (5, 4)), zorder=2)
    ax.text(ns[-1] + 0.4, ALPHA * 1.5, "α = 0.05", va='bottom', ha='right',
            fontsize=9, color='red')
    for n, f in zip(ns, frac):
        ax.text(n, 1.5, f"{f*100:.0f}", ha='center', va='bottom', fontsize=7,
                color='black' if f >= 0.8 else '0.55')
    ax.text(1.0, 1.055, "% of random draws reaching p < 0.05", transform=ax.transAxes,
            ha='right', va='bottom', fontsize=8, color='0.35')
    ax.set_xlabel("Number of spots drawn at random (per condition)")
    ax.set_ylabel("p value of Welch's t-test")
    ax.set_xticks(ns); ax.set_xlim(ns[0] - 0.7, ns[-1] + 0.8)
    ax.set_ylim(FLOOR, 3.2); ax.set_yticks([1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1])
    ax.set_title(title, loc='left', pad=22)
    save(fig, stem)
    return fig


def subsampling_table(ns, dist, frac, stem):
    path = OUT_DIR / f"{stem}.csv"
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["n_spots", "pct_draws_significant", "median_p", "p_2.5pct", "p_97.5pct"])
        for n, d, f in zip(ns, dist, frac):
            lo, hi = np.percentile(d, [2.5, 97.5])
            w.writerow([n, f"{f*100:.1f}", f"{np.median(d):.3g}", f"{lo:.3g}", f"{hi:.3g}"])
    print("saved", path.name)
    return path


def first_n_reaching(ns, frac, level=0.95):
    for n, f in zip(ns, frac):
        if f >= level:
            return n
    return None

print(f"subsampling helpers ready (DRAWS = {DRAWS})")

In [ ]:
"""Supplementary Figure S5 - quenching: Fluorescent Probes vs Fluorescent Probes + Quencher Probes."""
try:
    ns5, dist5, frac5 = subsample_pvalues(d4b["FP"], d4b["FP+QP"])
    subsampling_table(ns5, dist5, frac5, "figS5_subsampling_FP_vs_FPQP")
    subsampling_panel(ns5, dist5, frac5,
                      "Fluorescent Probes  vs  Fluorescent Probes + Quencher Probes",
                      "figS5_subsampling_FP_vs_FPQP"); plt.show()
    RESULTS["S5"] = dict(ns=ns5, frac=frac5, n95=first_n_reaching(ns5, frac5))
    print(f"full-data p = {welch(d4b['FP'], d4b['FP+QP'])[2]:.3g};  "
          f"95% of draws significant from n = {RESULTS['S5']['n95']} spots")
except (NameError, KeyError) as e:
    print("SKIPPED Supplementary Figure S5 - run the Figure 4b cell first -", e)

In [ ]:
"""Supplementary Figure S6 - target recovery: FP+QP vs FP+QP+Target."""
try:
    ns6, dist6, frac6 = subsample_pvalues(d4b["FP+QP"], d4b["FP+QP+Target"])
    subsampling_table(ns6, dist6, frac6, "figS6_subsampling_FPQP_vs_target")
    subsampling_panel(ns6, dist6, frac6,
                      "Fluorescent Probes + Quencher Probes  vs  + Target",
                      "figS6_subsampling_FPQP_vs_target", colour='#C9BBD8'); plt.show()
    RESULTS["S6"] = dict(ns=ns6, frac=frac6, n95=first_n_reaching(ns6, frac6))
    print(f"full-data p = {welch(d4b['FP+QP'], d4b['FP+QP+Target'])[2]:.3g};  "
          f"95% of draws significant from n = {RESULTS['S6']['n95']} spots")
except (NameError, KeyError) as e:
    print("SKIPPED Supplementary Figure S6 - run the Figure 4b cell first -", e)

### Companion analysis (optional) — precision vs number of spots

Analytical counterpart to S5/S6: the 95 % confidence half-width of a condition mean and of a
fold change, from the CVs measured in Figure 4b.

    half-width(mean)  = t(0.975, n-1) · CV / sqrt(n)
    half-width(ratio) = t(0.975, n-1) · sqrt(CV_a²/n + CV_b²/n)

In [ ]:
def t_crit(df, q=0.975, lo=0.0, hi=100.0, it=200):
    """Inverse t by bisection on the two-tailed tail probability (no scipy)."""
    target = 2 * (1 - q)
    for _ in range(it):
        mid = 0.5 * (lo + hi)
        if t_sf2(mid, df) > target: lo = mid
        else: hi = mid
    return 0.5 * (lo + hi)

try:
    TARGET_PRECISION, NMAX = 30.0, 40
    cv = {k: v.std(ddof=1) / v.mean() for k, v in d4b.items()}
    print("CV per condition:", {k: f"{100*c:.0f}%" for k, c in cv.items()})

    ns = np.arange(3, NMAX + 1)
    tc = np.array([t_crit(n - 1) for n in ns])
    fig, ax = plt.subplots(figsize=(5.6, 3.9))
    for k, col in zip(["FP", "FP+QP", "FP+QP+Target"], [NPG[3], NPG[5], NPG[0]]):
        ax.plot(ns, 100 * tc * cv[k] / np.sqrt(ns), color=col, lw=1.4, label=f"mean, {k}")
    ratio_hw = 100 * tc * np.sqrt(cv["FP+QP"]**2 / ns + cv["FP+QP+Target"]**2 / ns)
    ax.plot(ns, ratio_hw, color='black', lw=1.6, ls=(0, (4, 2)), label="fold change (+target / quenched)")
    ax.axhline(TARGET_PRECISION, color='0.5', lw=0.9, ls=':')
    ax.text(NMAX, TARGET_PRECISION * 1.05, f"±{TARGET_PRECISION:.0f}%", ha='right',
            va='bottom', fontsize=8, color='0.4')
    ax.set_xlabel("Number of spots per condition")
    ax.set_ylabel("95% CI half-width (% of the value)")
    ax.set_ylim(0, 120); ax.legend(frameon=False, fontsize=8)
    save(fig, "figS_precision_vs_spotnumber"); plt.show()
except (NameError, KeyError) as e:
    print("SKIPPED precision curve - run the Figure 4b cell first -", e)

---
## S8 · Supplementary Figure S8 — pumping characterization

Displaced volume per pumping cycle versus valve actuation time at 45 kPa.
`EAGER Data (from Prism).xlsx` → sheet `Volume per cycle nL`
(columns: X = actuation time in ms, then replicates r1–r3 and a Prism-computed mean).
Bars are mean ± SEM of the three replicates, matching the caption.

This panel was originally drawn in GraphPad Prism (`Volume_per_cycle_vs_time_ms.svg`);
the cell below reproduces it in matplotlib so the whole figure set comes from one place.

In [ ]:
try:
    rows = load_sheet("EAGER Data (from Prism).xlsx", "Volume per cycle nL")
    hdr_i = next(i for i, r in enumerate(rows) if str(r[0]).strip() == "X")
    data = [r for r in rows[hdr_i + 1:] if isinstance(r[0], (int, float))]
    x = np.array([r[0] for r in data], float)                       # actuation time (ms)
    reps = np.array([[c for c in r[2:5]] for r in data], float)     # three replicates
    mean = reps.mean(1); sem = reps.std(1, ddof=1) / np.sqrt(reps.shape[1])
    OPT = 600.0                                                     # chosen setpoint (ms)

    fig, ax = plt.subplots(figsize=(5.6, 3.8))
    ax.bar(x, mean, width=70, color=NPG[1], edgecolor='black', linewidth=0.6, yerr=sem,
           error_kw=dict(elinewidth=0.9, capsize=2.5, capthick=0.9, ecolor='black'), zorder=3)
    for k in range(reps.shape[1]):
        ax.scatter(x, reps[:, k], s=9, color='black', alpha=0.45, zorder=4, linewidths=0)
    ax.add_patch(plt.Rectangle((OPT - 55, 0), 110, max(mean + sem) * 1.12, fill=False,
                               edgecolor=NPG[0], lw=1.2, ls=(0, (4, 3)), zorder=5))
    ax.text(OPT, max(mean + sem) * 1.14, "optimal\nactuation time", ha='center', va='bottom',
            fontsize=8, color=NPG[0])
    ax.set_xlabel("Valve actuation time (ms)")
    ax.set_ylabel("Volume per cycle (µL)")
    ax.set_ylim(0, max(mean + sem) * 1.35)
    ax.set_xticks(x[::2])
    i_opt = int(np.argmin(abs(x - OPT)))
    print(f"at {OPT:.0f} ms: {mean[i_opt]:.3f} ± {sem[i_opt]:.3f} µL per cycle "
          f"({1000*mean[i_opt]:.0f} nL), n = {reps.shape[1]} replicates")
    RESULTS["S8"] = dict(x=x, mean=mean, sem=sem)
    save(fig, "figS8_volume_per_cycle"); plt.show()
except (MissingInput, StopIteration) as e:
    print("SKIPPED Supplementary Figure S8 -", e)

> **Unit check:** the sheet named "Volume per cycle nL" holds values of 0.23–0.65, and the
> Results text quotes "about 600 nL per cycle" at 600 ms — so the numbers are **µL**, not nL.
> Relabel the sheet or the axis before submission.

---
## 9 · Consolidated statistics table and reproducibility check

Writes one CSV holding every number quoted for Figures 4 and 5, then compares the values just
computed against those recorded when the panels were first generated (2026-09-02).

In [ ]:
def write_stats_table(path=None):
    path = Path(path or OUT_DIR / "fig4_fig5_stats.csv")
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["panel", "comparison", "group", "n", "mean_au", "sd_au",
                    "fold_or_retention", "test", "t", "df", "p", "stars"])
        if "4b" in RESULTS:
            r = RESULTS["4b"]
            for key, (a, b) in {"FP_vs_FPQP": ("FP", "FP+QP"),
                                "FPQP_vs_T":  ("FP+QP", "FP+QP+Target"),
                                "FP_vs_T":    ("FP", "FP+QP+Target")}.items():
                s = r[key]
                for g in (a, b):
                    d = r["desc"][g]
                    w.writerow(["4b", key, g, d["n"], f"{d['mean']:.1f}", f"{d['sd']:.1f}", "",
                                "Welch", f"{s['t']:.3f}", f"{s['df']:.1f}", f"{s['p']:.3g}",
                                s["stars"]])
        if "4d" in RESULTS:
            r = RESULTS["4d"]
            for name, s in (("Welch", r["welch"]), ("paired", r["paired"])):
                for g in ("before", "after"):
                    d = r["desc"][g]
                    w.writerow(["4d", "before_vs_after", g, d["n"], f"{d['mean']:.1f}",
                                f"{d['sd']:.1f}", f"{r['retention_perspot_pct']:.1f}% retained",
                                name, f"{s['t']:.3f}", f"{s['df']:.1f}", f"{s['p']:.3g}",
                                s["stars"]])
        for panel in ("5b", "5d"):
            if panel not in RESULTS: continue
            for t, s in RESULTS[panel].items():
                for g, d in (("no target", s["no_target"]), ("+ target", s["target"])):
                    w.writerow([panel, f"{t} no_target_vs_target", g, d["n"],
                                f"{d['mean']:.1f}", f"{d['sd']:.1f}", f"{s['fold']:.2f}x",
                                "Welch", f"{s['t']:.3f}", f"{s['df']:.1f}", f"{s['p']:.3g}",
                                s["stars"]])
    print("saved", path)
    return path

write_stats_table()

In [ ]:
"""Reproducibility check against the values recorded on 2026-09-02.

NOTE: all four source-data CSVs were regenerated after that snapshot (Fig 4b 2026-09-03,
Fig 4d 2026-09-04, Fig 5b/5d 2026-09-07) and the spot counts changed, so DIFF lines are
expected until REFERENCE is updated to the numbers that are actually going into the
manuscript. Decide which version is correct, then paste the new values in below.
"""
REFERENCE = {
    "fig4b": {"n": 17, "mean_FP": 19621, "mean_FPQP": 4860, "mean_FPQPT": 10072,
              "stars_FP_vs_FPQP": "****", "stars_FPQP_vs_T": "***", "stars_FP_vs_T": "**"},
    "fig4d": {"n": 31, "mean_before": 2874, "mean_after": 2685,
              "retention_pct": 92.3, "welch_stars": "n.s.", "paired_p": 0.009856},
    "fig5b": {"1 h": {"fold": 1.901, "p": 0.000448}, "24 h": {"fold": 1.970, "p": 3.29e-08}},
    "fig5d": {"1 h": {"fold": 6.421, "p": 2.5e-11},  "24 h": {"fold": 3.155, "p": 2.21e-12}},
}

def reproducibility_check(save_to=None):
    lines, ok = [], True

    def cmp(label, got, want, tol, fmt="{:.4g}"):
        nonlocal ok
        if isinstance(want, str):
            good = (got == want); detail = f"got {got}, recorded {want}"
        else:
            good = abs(got - want) <= tol * max(1.0, abs(want))
            detail = f"got {fmt.format(got)}, recorded {fmt.format(want)}"
        ok &= good
        lines.append(f"  [{'OK  ' if good else 'DIFF'}] {label}: {detail}")

    if "4b" in RESULTS:
        r, ref = RESULTS["4b"], REFERENCE["fig4b"]
        lines.append("Figure 4b")
        cmp("n (FP)", r["desc"]["FP"]["n"], ref["n"], 0)
        cmp("mean FP", r["desc"]["FP"]["mean"], ref["mean_FP"], 0.001)
        cmp("mean FP+QP", r["desc"]["FP+QP"]["mean"], ref["mean_FPQP"], 0.001)
        cmp("mean FP+QP+Target", r["desc"]["FP+QP+Target"]["mean"], ref["mean_FPQPT"], 0.001)
        cmp("stars FP vs FP+QP", r["FP_vs_FPQP"]["stars"], ref["stars_FP_vs_FPQP"], 0)
        cmp("stars FP+QP vs Target", r["FPQP_vs_T"]["stars"], ref["stars_FPQP_vs_T"], 0)
        cmp("stars FP vs Target", r["FP_vs_T"]["stars"], ref["stars_FP_vs_T"], 0)
    if "4d" in RESULTS:
        r, ref = RESULTS["4d"], REFERENCE["fig4d"]
        lines.append("Figure 4d")
        cmp("n", r["desc"]["before"]["n"], ref["n"], 0)
        cmp("mean before", r["desc"]["before"]["mean"], ref["mean_before"], 0.001)
        cmp("mean after", r["desc"]["after"]["mean"], ref["mean_after"], 0.001)
        cmp("per-spot retention %", r["retention_perspot_pct"], ref["retention_pct"], 0.005)
        cmp("Welch stars (reported)", r["welch"]["stars"], ref["welch_stars"], 0)
        cmp("paired p", r["paired"]["p"], ref["paired_p"], 0.02)
    for panel, key in (("5b", "fig5b"), ("5d", "fig5d")):
        if panel not in RESULTS: continue
        lines.append(f"Figure {panel}")
        for t in ("1 h", "24 h"):
            cmp(f"{t} fold", RESULTS[panel][t]["fold"], REFERENCE[key][t]["fold"], 0.005)
            cmp(f"{t} Welch p", RESULTS[panel][t]["p"], REFERENCE[key][t]["p"], 0.05)

    body = ("REPRODUCIBILITY CHECK - recomputed vs values recorded 2026-09-02.\n"
            "p values are computed without scipy, so ~2 significant figures of agreement\n"
            "is expected rather than bit-identical output.\n\n"
            + "\n".join(lines)
            + f"\n\nRESULT: {'all values reproduce' if ok else 'DIFFERENCES FOUND - see [DIFF] lines'}\n")
    print(body)
    path = Path(save_to or OUT_DIR / "reproducibility_check.txt")
    path.write_text(body)
    return ok

reproducibility_check()